In [10]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline

In [4]:
results = pd.read_parquet('../data/interim/results_clean.parquet')

In [7]:
# Data scope: drop pre-1970 (sparse, defunct-team-heavy, near-zero-weight under decay anyway)
# Applied here at modeling time - the cleaned parquet stays a full history
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

print(f'Before cut: {len(results)}')
print(f'After 1970 cut: {len(model_df)}')
print(f'Dropped: {len(results) - len(model_df)}')
print(f"Date range: {model_df['date'].min().date()} → {model_df['date'].max().date()}")

Before cut: 49482
After 1970 cut: 41522
Dropped: 7960
Date range: 1970-01-04 → 2026-06-30


In [9]:
home_rows = model_df.rename(columns={
    'home_team': 'attacking_team',
    'away_team': 'defending_team',
    'home_score': 'goals'
})[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

home_rows['is_home'] = (~home_rows['neutral']).astype(int) # home advantage only when not neutral

away_rows = model_df.rename(columns={
    'away_team': 'attacking_team',
    'home_team': 'defending_team',
    'away_score': 'goals'
})[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

away_rows['is_home'] = 0 # away side never gets home advantage

long_df = pd.concat([home_rows, away_rows], ignore_index=True)
print(long_df.shape)
print(long_df.head())

(83044, 6)
        date attacking_team  defending_team  goals  neutral  is_home
0 1970-01-04          Malta      Luxembourg      1    False        1
1 1970-01-14        England     Netherlands      0    False        1
2 1970-01-28         Israel     Netherlands      0    False        1
3 1970-02-04           Peru  Czechoslovakia      0    False        1
4 1970-02-06       Cameroon     Ivory Coast      3     True        0
